# Notebook 05 — Детекция объектов на дороге (YOLOv8)

## Что делаем:
1. Устанавливаем YOLOv8 (ultralytics)
2. Загружаем предобученную модель YOLOv8n (nano — быстрая)
3. Детектируем объекты на кадрах из датасета Udacity
4. Рисуем bounding boxes с классами и уверенностью
5. Анализируем какие объекты встречаются на дороге
6. Объединяем PilotNet + YOLO в единый пайплайн

---
**Классы для детекции:** автомобиль, человек, светофор, знак остановки

> YOLOv8 обучен на COCO (80 классов). Мы используем предобученные веса — дообучение не нужно.

In [ ]:
!pip install -q ultralytics torch torchvision opencv-python numpy matplotlib

from google.colab import drive
drive.mount('/content/drive')

import sys, os, json
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

with open('/content/drive/MyDrive/diploma/config.json') as f:
    cfg = json.load(f)

PROJECT_DIR = cfg['project_dir']
OUTPUTS_DIR = cfg['outputs_dir']
MODELS_DIR  = cfg['models_dir']
CSV_PATH    = cfg['csv_path']
IMG_DIR     = cfg['img_dir']

src_path = os.path.join(PROJECT_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {device}')

In [ ]:
import subprocess
result = subprocess.run(['git', '-C', '/content/drive/MyDrive/diploma', 'pull'], capture_output=True, text=True)
print(result.stdout or result.stderr)

## Шаг 1. Загрузка YOLOv8

**YOLOv8n (nano)** — самая лёгкая версия, идеальна для реального времени:  
- 3.2M параметров  
- ~6ms на кадр (GPU)  
- Веса загружаются автоматически (~6 MB)

In [ ]:
from ultralytics import YOLO

# Загружаем YOLOv8n с предобученными весами COCO
yolo_model = YOLO('yolov8n.pt')

print('✓ YOLOv8n загружен')
print(f'  Классов в COCO: 80')

# Классы которые нас интересуют (индексы в COCO)
CLASSES_OF_INTEREST = {
    0:  'person',        # пешеход
    2:  'car',           # автомобиль
    7:  'truck',         # грузовик
    9:  'traffic light', # светофор
    11: 'stop sign',     # знак СТОП
}

# Цвета для каждого класса (BGR для OpenCV)
CLASS_COLORS = {
    0:  (0, 255, 0),     # пешеход — зелёный
    2:  (255, 100, 0),   # автомобиль — синий
    7:  (0, 100, 255),   # грузовик — оранжевый
    9:  (0, 255, 255),   # светофор — жёлтый
    11: (0, 0, 255),     # знак СТОП — красный
}

print('\nОтслеживаемые классы:')
for cls_id, name in CLASSES_OF_INTEREST.items():
    print(f'  [{cls_id:2d}] {name}')

## Шаг 2. Детекция на одном изображении

In [ ]:
import pandas as pd

def detect_and_draw(img_bgr, conf_threshold=0.3):
    """
    Запускает YOLO на изображении и рисует bounding boxes.

    Параметры
    ----------
    img_bgr        : np.ndarray — изображение в формате BGR
    conf_threshold : float — минимальная уверенность для отображения

    Возвращает
    ----------
    tuple(np.ndarray, list)
        Изображение с boxes, список обнаруженных объектов
    """
    results = yolo_model(img_bgr, conf=conf_threshold, verbose=False)[0]

    output_img  = img_bgr.copy()
    detections  = []

    for box in results.boxes:
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])

        # Фильтруем только интересующие классы
        if cls_id not in CLASSES_OF_INTEREST:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        label  = CLASSES_OF_INTEREST[cls_id]
        color  = CLASS_COLORS.get(cls_id, (200, 200, 200))

        # Рисуем прямоугольник
        cv2.rectangle(output_img, (x1, y1), (x2, y2), color, 2)

        # Подпись с классом и уверенностью
        label_text = f'{label} {conf:.2f}'
        (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(output_img, (x1, y1 - th - 6), (x1 + tw + 4, y1), color, -1)
        cv2.putText(output_img, label_text, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        detections.append({'class': label, 'conf': conf,
                            'bbox': (x1, y1, x2, y2)})

    return output_img, detections


# Тест на одном изображении из датасета
df = pd.read_csv(CSV_PATH, header=None,
                 names=['center','left','right','steering','throttle','reverse','speed'])

# Берём несколько случайных кадров
sample_rows = df.sample(9, random_state=99)

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
fig.suptitle('YOLOv8n — детекция объектов на кадрах Udacity',
             fontsize=13, fontweight='bold')

total_detections = {}

for ax, (_, row) in zip(axes.flatten(), sample_rows.iterrows()):
    img_path = os.path.join(IMG_DIR, os.path.basename(str(row['center']).strip()))

    if not os.path.exists(img_path):
        ax.text(0.5, 0.5, 'Файл не найден', ha='center', va='center')
        ax.axis('off')
        continue

    img = cv2.imread(img_path)
    result_img, dets = detect_and_draw(img, conf_threshold=0.3)
    result_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

    ax.imshow(result_rgb)

    # Заголовок с найденными объектами
    if dets:
        found = ', '.join(set(d['class'] for d in dets))
        ax.set_title(f'Найдено: {found}', fontsize=8, color='green')
        for d in dets:
            total_detections[d['class']] = total_detections.get(d['class'], 0) + 1
    else:
        ax.set_title('Объекты не найдены', fontsize=8, color='gray')

    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'yolo_detections_grid.png'), dpi=150)
plt.show()

print('\nОбъекты, найденные в выборке:')
for cls, count in sorted(total_detections.items(), key=lambda x: -x[1]):
    print(f'  {cls:<15}: {count}')

## Шаг 3. Статистика детекций на большой выборке

In [ ]:
# Анализируем 100 случайных кадров
from tqdm import tqdm

N_SAMPLES = 100
sample_df = df.sample(N_SAMPLES, random_state=42)

stats = {name: 0 for name in CLASSES_OF_INTEREST.values()}
n_frames_with_objects = 0

for _, row in tqdm(sample_df.iterrows(), total=N_SAMPLES, desc='Детекция'):
    img_path = os.path.join(IMG_DIR, os.path.basename(str(row['center']).strip()))
    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    _, dets = detect_and_draw(img, conf_threshold=0.3)

    if dets:
        n_frames_with_objects += 1
        for d in dets:
            if d['class'] in stats:
                stats[d['class']] += 1

print(f'\nРезультаты на {N_SAMPLES} кадрах:')
print(f'  Кадров с объектами: {n_frames_with_objects} ({n_frames_with_objects/N_SAMPLES*100:.1f}%)')
print('\nЧастота объектов:')
for cls, count in sorted(stats.items(), key=lambda x: -x[1]):
    pct = count / N_SAMPLES * 100
    print(f'  {cls:<15}: {count:3d} обнаружений ({pct:.1f}% кадров)')

# График
fig, ax = plt.subplots(figsize=(9, 4))
classes = list(stats.keys())
counts  = list(stats.values())
colors  = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']

bars = ax.bar(classes, counts, color=colors[:len(classes)], edgecolor='white')
ax.bar_label(bars, padding=3)
ax.set_ylabel('Количество обнаружений', fontsize=11)
ax.set_title(f'Статистика детекций YOLOv8n ({N_SAMPLES} кадров)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'yolo_statistics.png'), dpi=150)
plt.show()

## Шаг 4. Объединённый пайплайн: PilotNet + YOLO

Демонстрируем полную систему:  
- YOLOv8 определяет объекты на дороге  
- PilotNet предсказывает угол руля  
- Результат отображается на кадре

In [ ]:
from model   import get_model
from dataset import preprocess_image

# Загружаем PilotNet
model_path = os.path.join(MODELS_DIR, 'best_model.pth')
pilot_model = get_model(device=device)
pilot_model.load_state_dict(torch.load(model_path, map_location=device, weights_only=False))
pilot_model.eval()
print('PilotNet загружен')


def predict_steering(img_bgr, model, device):
    """
    Предсказывает угол руля по изображению.
    Возвращает float — угол в диапазоне [-1, 1].
    """
    img_processed = preprocess_image(img_bgr)
    tensor = torch.tensor(img_processed).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        angle = model(tensor).item()
    return angle


def draw_steering_indicator(img, angle, center_x=None, center_y=None):
    """
    Рисует стрелку направления руля на изображении.
    """
    h, w = img.shape[:2]
    cx = center_x or w // 2
    cy = center_y or h - 30

    arrow_len = 60
    import math
    rad = math.pi / 2 - angle * math.pi / 2
    ex  = int(cx + arrow_len * math.cos(rad))
    ey  = int(cy - arrow_len * math.sin(rad))

    color = (0, 255, 0) if abs(angle) < 0.1 else (0, 165, 255)
    cv2.arrowedLine(img, (cx, cy), (ex, ey), color, 3, tipLength=0.3)

    return img


def full_pipeline(img_bgr, pilot_model, device):
    """
    Полный пайплайн: YOLO детекция + PilotNet предсказание.
    Возвращает аннотированное изображение.
    """
    # 1. Детекция объектов (YOLO)
    result_img, detections = detect_and_draw(img_bgr, conf_threshold=0.3)

    # 2. Предсказание угла руля (PilotNet)
    steering_angle = predict_steering(img_bgr, pilot_model, device)

    # 3. Рисуем стрелку угла руля
    draw_steering_indicator(result_img, steering_angle)

    # 4. Подпись угла
    direction = 'PRYAMO' if abs(steering_angle) < 0.05 else \
                ('VPRAVO' if steering_angle > 0 else 'VLEVO')
    text = f'Rul: {steering_angle:+.3f} ({direction})'
    cv2.putText(result_img, text, (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 0), 2)

    # 5. Счётчик объектов
    if detections:
        obj_text = f'Obektov: {len(detections)}'
        cv2.putText(result_img, obj_text, (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

    return result_img, steering_angle, detections


print('Funktsii pipelaina opredeleny')

In [ ]:
# Запускаем пайплайн на 6 случайных кадрах
pipeline_samples = df.sample(6, random_state=55)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
fig.suptitle('Объединённый пайплайн: PilotNet (угол руля) + YOLOv8 (объекты)',
             fontsize=12, fontweight='bold')

for ax, (_, row) in zip(axes.flatten(), pipeline_samples.iterrows()):
    img_path = os.path.join(IMG_DIR, os.path.basename(str(row['center']).strip()))

    if not os.path.exists(img_path):
        ax.axis('off'); continue

    img = cv2.imread(img_path)
    result, angle, dets = full_pipeline(img, pilot_model, device)
    result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

    real_angle = float(row['steering'])

    ax.imshow(result_rgb)
    ax.set_title(
        f'Пред: {angle:+.3f} | Реал: {real_angle:+.3f}\n'
        f'Объектов: {len(dets)}',
        fontsize=9
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'full_pipeline.png'), dpi=150)
plt.show()

## Итог

✅ YOLOv8n успешно детектирует объекты на кадрах Udacity  
✅ Статистика объектов собрана  
✅ Пайплайн PilotNet + YOLO работает и визуализирован  
✅ Все результаты сохранены в `outputs/`  

---

## Все файлы проекта

| Файл | Описание |
|------|----------|
| `outputs/data_analysis.png` | Распределение углов руля в датасете |
| `outputs/preprocessing_steps.png` | Шаги предобработки изображения |
| `outputs/augmentation_examples.png` | Виды аугментации |
| `outputs/balance_comparison.png` | Балансировка датасета |
| `outputs/training_curves.png` | Кривые обучения |
| `outputs/pred_vs_actual.png` | Предсказания vs реальность |
| `outputs/error_distribution.png` | Распределение ошибок |
| `outputs/sample_predictions.png` | Примеры предсказаний (3×3) |
| `outputs/yolo_detections_grid.png` | Детекция YOLO (сетка) |
| `outputs/full_pipeline.png` | Полный пайплайн |
| `models/best_model.pth` | Веса лучшей модели |